# 01_ingest_raw.py 결과 확인

`data/raw` 아래 companies / prices / financials / dividends 원본 수집 결과를 확인한다.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.master("spark://spark-master:7077").appName("check_raw").getOrCreate()

# .cache()로 메모리에 적재 - 이후 셀들이 매번 parquet을 재스캔하지 않고 캐시를 재사용해 빨라짐
companies = spark.read.parquet("/opt/spark-apps/data/raw/companies").cache()
prices = spark.read.parquet("/opt/spark-apps/data/raw/prices").cache()
financials = spark.read.parquet("/opt/spark-apps/data/raw/financials").cache()
dividends = spark.read.parquet("/opt/spark-apps/data/raw/dividends").cache()

print(f"companies : {companies.count()}건")
print(f"prices    : {prices.count()}건")
print(f"financials: {financials.count()}건")
print(f"dividends : {dividends.count()}건")

companies : 2555건
prices    : 35692건
financials: 18142건
dividends : 1298건


## 1. companies (KRX 상장종목 + DART corpCode 매핑)

In [3]:
companies.orderBy("stock_code").toPandas()

,corp_cls,corp_code,corp_name,stock_code,bas_dt
0,Y,00119195,동화약품,000020,20260724
1,Y,00112378,KR모터스,000040,20260724
2,Y,00101628,경방,000050,20260724
3,Y,00126937,삼양홀딩스,000070,20260724
4,Y,00150244,하이트진로,000080,20260724
...,...,...,...,...,...
2550,K,01416235,고스트스튜디오,950190,20260724
2551,K,01442115,소마젠,950200,20260724
2552,Y,01510489,프레스티지바이오파마,950210,20260724
2553,K,01511558,네오이뮨텍,950220,20260724


## 2. prices (날짜별 건수, snapshot_type별 분포)
`snapshot_type`: current(백테스트 대상 구간) / 1m_ago / 12m_ago(모멘텀 계산용 스냅샷)

In [4]:
prices.groupBy("snapshot_type", "bas_dt").count().orderBy("snapshot_type", "bas_dt").toPandas()

,snapshot_type,bas_dt,count
0,12m_ago,20250729,2492
1,1m_ago,20260629,2550
2,current,20260710,2553
3,current,20260713,2554
4,current,20260714,2554
5,current,20260715,2554
6,current,20260716,2554
7,current,20260720,2554
8,current,20260721,2554
9,current,20260722,2554


In [5]:
# 특정 종목 시세 흐름 확인 (종목코드 바꿔가며 조회)
prices.filter(F.col("stock_code") == "005930").orderBy("snapshot_type", "bas_dt").toPandas()

,bas_dt,close_price,fluctuation_rate,listed_share_count,market_cap,open_price,snapshot_type,stock_code,year
0,20250729,70600,0.28,5919637922,417926437293200,70800,12m_ago,005930,2025
1,20260629,323000,-4.86,5846278608,1888347990384000,331000,1m_ago,005930,2026
2,20260710,285000,2.52,5846278608,1666189403280000,291000,current,005930,2026
3,20260713,254500,-10.70,5846278608,1487877905736000,285000,current,005930,2026
4,20260714,263000,3.34,5846278608,1537571273904000,255000,current,005930,2026
5,20260715,279500,6.27,5846278608,1634034870936000,283500,current,005930,2026
6,20260716,255000,-8.77,5846278608,1490801045040000,264500,current,005930,2026
7,20260720,244000,-4.31,5846278608,1426491980352000,241000,current,005930,2026
8,20260721,259000,6.15,5846278608,1514186159472000,247000,current,005930,2026
9,20260722,260500,0.58,5846278608,1522955577384000,276000,current,005930,2026


## 3. financials (재무제표, --dart-limit 적용 종목만 존재)

In [6]:
print(f"재무제표 보유 종목 수: {financials.select('stock_code').distinct().count()}개")
print(f"전체 상장 종목 수: {companies.select('stock_code').distinct().count()}개")

financials.groupBy("fs_div", "currency").count().orderBy("fs_div", "currency").toPandas()

재무제표 보유 종목 수: 86개
전체 상장 종목 수: 2555개


,fs_div,currency,count
0,CFS,CNY,953
1,CFS,KRW,15675
2,OFS,KRW,1514


In [7]:
# 특정 종목 재무제표 항목 확인 (종목코드 바꿔가며 조회)
financials.filter(F.col("stock_code") == "000020") \
    .select("account_id", "account_nm", "sj_nm", "thstrm_nm", "thstrm_amount", "frmtrm_nm", "frmtrm_amount") \
    .toPandas()

,account_id,account_nm,sj_nm,thstrm_nm,thstrm_amount,frmtrm_nm,frmtrm_amount
0,dart_CashAndCashEquivalentsAtBeginningOfPeriodCf,기초현금및현금성자산,현금흐름표,제 95 기,34347103567,제 94 기,60479642642
1,dart_CashAndCashEquivalentsAtEndOfPeriodCf,기말현금및현금성자산,현금흐름표,제 95 기,72317114790,제 94 기,34347103567
2,ifrs-full_CashFlowsFromUsedInFinancingActivities,재무활동현금흐름,현금흐름표,제 95 기,5865503201,제 94 기,-12013033236
3,dart_PaymentsOfFinanceLeaseLiabilitiesClassifi...,리스부채의 지급,현금흐름표,제 95 기,2652253399,제 94 기,2030789836
4,dart_ProceedsFromLongTermBorrowings,장기차입금의 증가,현금흐름표,제 95 기,16000000000,제 94 기,0
...,...,...,...,...,...,...,...
283,ifrs-full_ProfitLoss,당기순이익(손실),포괄손익계산서,제 95 기,28237045586,제 94 기,21591210925
284,ifrs-full_ProfitLossAttributableToNoncontrolli...,비지배지분,포괄손익계산서,제 95 기,798755825,제 94 기,1207751952
285,ifrs-full_ProfitLossAttributableToOwnersOfParent,지배기업 소유주지분,포괄손익계산서,제 95 기,27438289761,제 94 기,20383458973
286,ifrs-full_ProfitLossBeforeTax,법인세비용차감전순이익(손실),포괄손익계산서,제 95 기,36955579790,제 94 기,23085548187


## 4. dividends (배당, --dart-limit 적용 종목만 존재)

In [8]:
dividends.groupBy("se", "stock_knd").count().orderBy("se", "stock_knd").toPandas()

,se,stock_knd,count
0,(별도)당기순이익(백만원),-,85
1,(연결)당기순이익(백만원),-,85
2,(연결)주당순이익(원),-,85
3,(연결)현금배당성향(%),-,85
4,주당 주식배당(주),-,69
5,주당 주식배당(주),2우선주식,1
6,주당 주식배당(주),3우선주식,1
7,주당 주식배당(주),4우선주,1
8,주당 주식배당(주),4우선주식,1
9,주당 주식배당(주),보통주,55


In [10]:
spark.stop()